In [109]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub


In [135]:
path = kagglehub.dataset_download("gowtha69/ovarian-cancer-risk-and-detection-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\SHALINE\.cache\kagglehub\datasets\gowtha69\ovarian-cancer-risk-and-detection-dataset\versions\1


In [168]:
df = pd.read_csv(path+"/oc_risk50k_single_dataset.csv")

In [149]:
df = df.drop(columns=["CA125", "HE4", "Albumin", "Imaging_Risk_Score"])

In [150]:
train_df = df[df["Partition"] == "Train"].copy()
X_train = train_df.iloc[: ,1:45].values.astype(float)

In [151]:
y_train = train_df["Ovarian_Cancer_Status"].values.astype(float)
print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1
X_train = (X_train - mean) / std

beta0= 0.0
beta = np.zeros(X_train.shape[1])
epochs = 1000
learning_rate = 0.01

X_train shape:  (34946, 44)
y_train shape:  (34946,)


In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def log_loss(y_train, pred):
    return -np.mean((y_train * np.log(pred)) + ((1 - y_train) * np.log(1 - pred))) 

for i in range(epochs):
    z = beta0 + np.dot(X_train, beta)
    pred = sigmoid(z)
    error = pred - y_train
    grad0 = np.mean(error)
    grad = np.dot(X_train.T, error) / len(y_train)

    beta0 = beta0 - learning_rate * grad0
    beta = beta - learning_rate  * grad

    if i % 100 == 0:
        eps = 1e-15
        pred = np.clip(pred, eps, 1 - eps)
        loss = log_loss(y_train, pred)
        print(f"Epoch: {i}")
        print(f"Loss: {loss:.6f}")
        print(f"Beta0: {beta0:.6f}")
        print(f"Max Beta: {np.max(np.abs(beta)):.6f}")
        print("-" * 40)
        print((pred >= 1).sum())
    

In [169]:

z = beta0 + np.dot(X_train, beta)
pred = sigmoid(z)

y_pred = (pred >= 0.5).astype(int)

loss = log_loss(y_train, pred)
accuracy = np.mean(y_pred == y_train) * 100

print("\n" + "=" * 60)
print("           LOGISTIC REGRESSION MODEL RESULTS")
print("=" * 60)

print("\nTRAINING INFORMATION")
print("-" * 60)
print(f"Epochs          : {epochs}")
print(f"Learning Rate   : {learning_rate}")
print(f"Final Loss      : {loss:.6f}")
print(f"Training Accuracy: {accuracy:.2f}%")

print("\nMODEL PARAMETERS")
print("-" * 60)
print(f"Intercept (Beta0): {beta0:.6f}")

print("\nFeature Coefficients:")
for i, b in enumerate(beta, start=1):
    print(f"Beta{i:<4}: {b:>12.6f}")

print("\nPREDICTION RESULTS")
print("-" * 60)
print(f"{'No.':<8}{'Actual':<12}{'Probability':<18}{'Predicted':<12}")
print("-" * 60)

for i in range(10):
    print(
        f"{i+1:<8}"
        f"{int(y_train[i]):<12}"
        f"{pred[i]:<18.6f}"
        f"{y_pred[i]:<12}"
    )

print("-" * 60)

# 4. Final summary
print("\nMODEL SUMMARY")
print("-" * 60)
print(f"Total observations : {len(y_train)}")
print(f"Correct predictions: {np.sum(y_pred == y_train)}")
print(f"Incorrect predictions: {np.sum(y_pred != y_train)}")
print(f"Accuracy           : {accuracy:.2f}%")

print("=" * 60)



           LOGISTIC REGRESSION MODEL RESULTS

TRAINING INFORMATION
------------------------------------------------------------
Epochs          : 1000
Learning Rate   : 0.01
Final Loss      : 0.019770
Training Accuracy: 99.99%

MODEL PARAMETERS
------------------------------------------------------------
Intercept (Beta0): 0.035605

Feature Coefficients:
Beta1   :     0.094075
Beta2   :     0.078622
Beta3   :    -0.002016
Beta4   :     0.077854
Beta5   :     0.117487
Beta6   :     0.128703
Beta7   :     0.115190
Beta8   :     0.063544
Beta9   :     0.000985
Beta10  :     0.003822
Beta11  :    -0.089333
Beta12  :    -0.058870
Beta13  :     0.091459
Beta14  :     0.010991
Beta15  :    -0.091757
Beta16  :     0.298925
Beta17  :     0.265104
Beta18  :     0.218278
Beta19  :     0.137756
Beta20  :     0.186599
Beta21  :     0.339069
Beta22  :     0.239333
Beta23  :     0.002393
Beta24  :     0.477113
Beta25  :    -0.349403
Beta26  :     0.418784
Beta27  :     0.279349
Beta28  :     0.35258

In [163]:
test_df = df[df["Partition"] == "Test"].copy()

X_test = test_df.iloc[: ,1:45].values.astype(float)
y_test = test_df["Ovarian_Cancer_Status"].values.astype(float)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)

X_test shape:  (7531, 44)
y_test shape:  (7531,)


In [164]:
X_test = (X_test - mean) / std

In [165]:
z_test = beta0 + np.dot(X_test, beta)
pred_test = sigmoid(z_test)
y_pred_test = (pred_test > 0.5).astype(int)

In [171]:
print("\nUNSEEN DATA PREDICTIONS")
print("=" * 55)

print(f"{'Actual':<15}{'Probability':<20}{'Predicted':<15}")
print("-" * 55)

for actual, probability, predicted in zip(
    y_test, pred_test, y_pred_test
):
    print(f"{int(actual):<15}{probability:<20.6f}{predicted:<15}")

print("=" * 55)


UNSEEN DATA PREDICTIONS
Actual         Probability         Predicted      
-------------------------------------------------------
1              0.989215            1              
0              0.004712            0              
1              0.981756            1              
1              0.992542            1              
1              0.995530            1              
0              0.006959            0              
0              0.004878            0              
0              0.014064            0              
0              0.004482            0              
1              0.998155            1              
0              0.019269            0              
1              0.989812            1              
0              0.010318            0              
1              0.995738            1              
1              0.996024            1              
0              0.006587            0              
0              0.061010            0              
0

In [172]:
accuracy = np.mean(y_pred_test == y_test) * 100

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 99.99%
